# arcroad — M2 access to hospitals, first real run (v0.1)

**Question.** From each 250 m square where people live, how much further is the nearest
hospital by road than it looks on a map, and how many people carry that extra distance?
Government and private hospitals are measured separately.

**Measure (design approved 27 Oct 2026).** Excess = driving distance to the nearest hospital
you can reach by road, minus the straight line to the nearest hospital on the map. Distance
only, driving only, never travel time.

**Rule 5, declared before this runs.**
- *Hypothesis:* squares beside a divided arterial carry more excess than comparable squares
  that are not.
- *Competing explanation:* excess is driven by distance from the core and thin map coverage.
- *Separation test:* compare adjacent and non-adjacent squares only inside matched strata
  (same 5 km ring, same road-density tertile). SUPPORTED only if adjacent is worse in at
  least two thirds of at least three comparable strata.

**Nothing here is published.** Every output is a candidate for checking. The code that does
the measuring is `pipeline/access.py` in the public repo, with tests; this notebook only
fetches data and calls it.

**What you do:** Runtime → Run all. Cell 4 asks you to upload the Telangana health-centres
CSV once (see the instructions printed there). At the end a zip downloads; send it to Claude.

## 1. Install and get the code

In [ ]:
!pip -q install osmium rasterio tabulate
!apt-get -qq install -y osmium-tool > /dev/null 2>&1
import os, sys, subprocess
REPO = "/content/arcroad"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/rrmlogs8896/arcroad.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
sys.path.insert(0, REPO)
from pipeline import access as A
from pipeline.pairing import load_network, build_graph, extract_date, commit_sha, ZONES, ARTERIAL
SHA = commit_sha(REPO)
print("arcroad", SHA)

## 2. Configuration

In [ ]:
STUDY = ZONES["hmda_all"]            # (min_lon, min_lat, max_lon, max_lat): the M1 study area
CLIP_BUFFER_DEG = 0.20               # clip the map wider than we measure, as in M1 (F-3)
CITY = (78.4867, 17.3850)            # ring centre, as in M1
EXTRACT_URL = "https://download.geofabrik.de/asia/india/southern-zone-latest.osm.pbf"
PBF_PATH  = "/content/southern-zone.osm.pbf"
CLIP_PATH = "/content/hyderabad_buffered.osm.pbf"
GHS_EPOCH = 2025                     # modelled estimate; labelled as such everywhere
OUT_DIR = "/content/m2_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("study area", STUDY, "| cell", A.CELL_M, "m | route limit", A.ROUTE_LIMIT_M, "m")

## 3. Road map: download, clip, build the graph (same engine as M1)

In [ ]:
if not (os.path.exists(CLIP_PATH) and os.path.getsize(CLIP_PATH) > 100_000):
    if not (os.path.exists(PBF_PATH) and os.path.getsize(PBF_PATH) > 1_000_000):
        print("downloading", EXTRACT_URL)
        if subprocess.call(["wget", "-q", "--show-progress", "-O", PBF_PATH, EXTRACT_URL]) != 0:
            raise SystemExit("Download failed.")
    x0, y0, x1, y1 = STUDY; b = CLIP_BUFFER_DEG
    r = subprocess.run(["osmium", "extract", "-b", f"{x0-b},{y0-b},{x1+b},{y1+b}", PBF_PATH,
                        "-o", CLIP_PATH, "--overwrite", "-s", "complete_ways"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise SystemExit("Clip failed.")
EXTRACT_DATE = extract_date(CLIP_PATH) or extract_date(PBF_PATH)
net = load_network(CLIP_PATH)
graph = build_graph(net.ways, net.coords)

import numpy as np, hashlib
labels, counts = np.unique(graph.comp_label, return_counts=True)
MAIN = graph.comp_label == labels[np.argmax(counts)]
GRAPH_HASH = hashlib.sha1(f"{graph.n}|{graph.G.nnz}|{float(graph.G.data.sum()):.3f}|{EXTRACT_DATE}".encode()).hexdigest()[:12]
print(f"extract {EXTRACT_DATE} | nodes {graph.n:,} | edges {graph.G.nnz:,} | main network {MAIN.mean():.1%} of nodes | graph {GRAPH_HASH}")

## 4. Hospitals

**Government layer:** the Telangana Health, Medical & Family Welfare Department list of health
centres (Open Data Telangana; Government Open Data License – India). Hospitals and community
health centres that are government-owned and functional; PHCs, UHCs, sub-centres, dispensaries
and AYUSH are left out of v1. **Private layer:** OpenStreetMap `amenity=hospital` not already
covered by a government facility.

If the CSV is not found, the cell explains how to get it and asks you to upload it.

In [ ]:
import glob, io, csv, re
cands = [p for p in glob.glob("/content/*.csv") if re.search(r"health|hospital|centre|center", p, re.I)]
if not cands:
    print('''Telangana health-centres CSV not found. To get it (one time):
  1. Open https://data.telangana.gov.in and search "health centres latitude longitude".
  2. Open the dataset "Details of Health Centres with Latitude and Longitude in Telangana".
  3. Download the CSV resource.
  4. Click "Choose Files" below and pick that CSV.''')
    from google.colab import files
    up = files.upload()
    cands = [f"/content/{n}" for n in up]
TG_CSV = cands[0]
rows = list(csv.DictReader(io.open(TG_CSV, encoding="utf-8-sig", errors="replace")))
cols = list(rows[0].keys())

def pick(*words, avoid=()):
    for c in cols:
        lc = c.lower()
        if any(w in lc for w in words) and not any(a in lc for a in avoid):
            return c
    return None

COL = {
    "name": pick("facility name", "name of", "name", avoid=("district", "taluk", "mandal", "state")),
    "type": pick("facility type", "type"),
    "own":  pick("ownership", "owner", "authority"),
    "status": pick("status", "functional"),
    "lat":  pick("latitude", "lat"),
    "lon":  pick("longitude", "long", "lon"),
}
print("file:", os.path.basename(TG_CSV), "| rows:", len(rows))
print("columns used:", COL)
assert COL["lat"] and COL["lon"] and COL["type"] and COL["own"], "Column detection failed: send Claude the column list above."

official, bad_xy = [], 0
for i, r in enumerate(rows):
    try:
        lat, lon = float(r[COL["lat"]]), float(r[COL["lon"]])
    except (TypeError, ValueError):
        bad_xy += 1; continue
    if not (15.5 <= lat <= 20.0 and 77.0 <= lon <= 81.5):   # Telangana, generously
        bad_xy += 1; continue
    official.append({"id": str(i), "name": r.get(COL["name"]) if COL["name"] else None,
                     "facility_type": r[COL["type"]], "ownership": r[COL["own"]],
                     "status": r.get(COL["status"], "") if COL["status"] else "", "lon": lon, "lat": lat})
print(f"usable rows {len(official):,} | rows with missing or out-of-state coordinates {bad_xy:,}")

import collections
t = collections.Counter((o["ownership"], o["facility_type"]) for o in official)
print("\nownership x facility type (top 25):")
for (own, ft), n in t.most_common(25):
    print(f"  {n:6,}  {own[:20]:20}  {ft}  {'<- government hospital layer' if A.is_govt_hospital({'ownership': own, 'status': 'Functional', 'facility_type': ft}) else ''}")

In [ ]:
import osmium

class Hosp(osmium.SimpleHandler):
    def __init__(self):
        super().__init__(); self.out = []; self.relations = 0
    def _tags(self, o):
        t = dict(o.tags)
        return t if (t.get("amenity") == "hospital" or t.get("healthcare") == "hospital") else None
    def node(self, n):
        t = self._tags(n)
        if t: self.out.append({"id": f"n{n.id}", "lon": n.location.lon, "lat": n.location.lat, "tags": dict(t, amenity="hospital")})
    def way(self, w):
        t = self._tags(w)
        if not t: return
        try:
            xs = [n.lon for n in w.nodes]; ys = [n.lat for n in w.nodes]
        except Exception:
            return
        self.out.append({"id": f"w{w.id}", "lon": sum(xs)/len(xs), "lat": sum(ys)/len(ys), "tags": dict(t, amenity="hospital")})
    def relation(self, r):
        if self._tags(r): self.relations += 1

h = Hosp(); h.apply_file(CLIP_PATH, locations=True)
osm_h = h.out
layers = A.hospital_layers(official, osm_h)
x0, y0, x1, y1 = STUDY; m = 0.15   # keep hospitals just outside the study area: people near the edge use them
def near_study(d): return x0 - m <= d["lon"] <= x1 + m and y0 - m <= d["lat"] <= y1 + m
layers = {k: [d for d in v if near_study(d)] for k, v in layers.items()}
print(f"OSM hospitals {len(osm_h):,} (+{h.relations} mapped as relations, not used)")
for k, v in layers.items():
    print(f"{k:10} {len(v):5,} hospitals in or near the study area  (official {sum(d['source']=='official' for d in v)}, osm {sum(d['source']=='osm' for d in v)})")

## 5. People: GHSL population grid (European Commission JRC)

GHS-POP R2023A, 3 arc-second (~90 m). Reuse is authorised with acknowledgement. The
2025 epoch is a **modelled estimate**, and every output says so.

In [ ]:
import rasterio, zipfile
from rasterio.windows import from_bounds

base = (f"https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/GHS_POP_GLOBE_R2023A/"
        f"GHS_POP_E{GHS_EPOCH}_GLOBE_R2023A_4326_3ss/V1-0/tiles/")
name = f"GHS_POP_E{GHS_EPOCH}_GLOBE_R2023A_4326_3ss_V1_0"
tif = None
for R, C in [(8, 26), (7, 26), (8, 25), (9, 26), (7, 25), (8, 27)]:      # 10-degree tiles; Hyderabad is R8 C26
    z = f"/content/{name}_R{R}_C{C}.zip"
    if not os.path.exists(z) and subprocess.call(["wget", "-q", "-O", z, base + os.path.basename(z)]) != 0:
        os.remove(z); continue
    with zipfile.ZipFile(z) as zf:
        t = [n for n in zf.namelist() if n.endswith(".tif")][0]; zf.extract(t, "/content")
    with rasterio.open(f"/content/{t}") as src:
        b = src.bounds
        if b.left <= STUDY[0] and b.right >= STUDY[2] and b.bottom <= STUDY[1] and b.top >= STUDY[3]:
            tif = f"/content/{t}"; print("tile", R, C, "covers the study area"); break
if not tif:
    raise SystemExit("No GHS-POP tile covers the study area: send Claude this message.")

with rasterio.open(tif) as src:
    win = from_bounds(*STUDY, transform=src.transform)
    arr = src.read(1, window=win)
    tr = src.window_transform(win)
    nodata = src.nodata
rr, cc = np.nonzero((arr > 0) & (arr != nodata if nodata is not None else True))
px_lon, px_lat = rasterio.transform.xy(tr, rr, cc)       # pixel centres
pop = arr[rr, cc].astype(float)

grid, steps = A.grid_cells(STUDY)
cell_pop, outside = A.assign_population(px_lon, px_lat, pop, STUDY, steps)
cells = [dict(c, pop=cell_pop[c["cell_id"]]) for c in grid if cell_pop.get(c["cell_id"], 0) > 0]
print(f"grid {len(grid):,} squares | populated {len(cells):,} | people {sum(c['pop'] for c in cells):,.0f} "
      f"(GHS-POP {GHS_EPOCH}, modelled) | outside grid {outside:,.0f}")

## 6. Measure: every populated square against each layer

In [ ]:
import time
records, dropped = [], {}
for layer, dests in layers.items():
    t0 = time.time()
    recs, drop = A.measure_layer(cells, layer, dests, graph.lon, graph.lat, graph.G, eligible=MAIN)
    for r in recs:
        r.update(extract_date=EXTRACT_DATE, graph_hash=GRAPH_HASH, commit=SHA,
                 population_source=f"GHS-POP E{GHS_EPOCH} R2023A 3ss (modelled)",
                 destination_source="Telangana health centres (GODL-India)" if layer == "government" else "OpenStreetMap (ODbL)")
    records += recs; dropped[layer] = drop
    st = collections.Counter(r["status"] for r in recs)
    print(f"{layer:10} {len(recs):,} squares in {time.time()-t0:.0f}s | {dict(st)} | hospitals too far from any road: {len(drop)}")

problems = [p for r in records for p in A.export_problems(r)]
assert not problems, f"provenance problems: {collections.Counter(problems).most_common(5)}"
print("every row carries its provenance")

## 7. How many people, how much extra (candidates, not findings)

In [ ]:
from tabulate import tabulate
def summ(rs): return A.people_summary(rs)
rows_out = []
for layer in layers:
    for zone in ("hmda_all", "core_city", "patancheru"):
        zx0, zy0, zx1, zy1 = ZONES[zone]
        rs = [r for r in records if r["layer"] == layer and zx0 <= r["lon"] <= zx1 and zy0 <= r["lat"] <= zy1]
        s = summ(rs); s.update(layer=layer, zone=zone); rows_out.append(s)
keys = ["layer", "zone", "people_total", "people_measured", "median_excess_m", "p90_excess_m",
        "pct_people_excess_ge_1000m", "pct_people_excess_ge_2000m", "pct_people_excess_ge_5000m", "people_unmeasured_by_status"]
print(tabulate([[r.get(k) for k in keys] for r in rows_out], headers=keys, floatfmt=".0f"))

## 8. Rule 5: is it the medians, or just distance and thin mapping?

In [ ]:
art_lines = []
for wid, nodes, fwd, bwd, hw, divided, lvl in net.ways:
    if hw in ARTERIAL and divided:
        pts = [net.coords[n] for n in nodes if n in net.coords]
        if len(pts) > 1: art_lines.append(pts)
art_pts = A.densify(art_lines, 50)
print(f"divided arterial ways {len(art_lines):,} -> {len(art_pts):,} points every 50 m")

verdicts = {}
for layer in layers:
    rs = [r for r in records if r["layer"] == layer]
    A.add_strata(rs, art_pts, graph.lon, graph.lat, centre=CITY)
    v = A.rule5_verdict(rs); verdicts[layer] = v
    print(f"\n{layer.upper()}: {v['verdict']}  (adjacent worse in {v['adjacent_worse_in']} of {v['comparable_strata']} comparable strata)")
    show = [r for r in v["rows"] if r["comparable"]]
    print(tabulate([[r["stratum"], r["adj_people"], r["adj_median_excess_m"], r["not_people"], r["not_median_excess_m"], r["gap_m"]] for r in show],
                   headers=["stratum", "adj people", "adj median", "other people", "other median", "gap m"], floatfmt=".0f"))

## 9. Verification sample and outputs

The worst squares (at least 100 people, one per 1 km so a list of places is not one place
repeated) go to the verification sample: ground drive or a private routing check, never cited.

In [ ]:
from pipeline.diagnostics import cluster_dedup
import pandas as pd, datetime, shutil
stamp = datetime.date.today().isoformat()
sample = []
for layer in layers:
    rs = [r for r in records if r["layer"] == layer and r["status"] == "measured" and r["pop"] >= 100]
    sample += cluster_dedup(rs, 1000, key="excess_m")[:30]

pd.DataFrame(records).to_csv(f"{OUT_DIR}/m2_hospital_cells_{stamp}.csv", index=False)
pd.DataFrame(rows_out).to_csv(f"{OUT_DIR}/m2_summary_{stamp}.csv", index=False)
pd.DataFrame(sample).to_csv(f"{OUT_DIR}/m2_verification_sample_{stamp}.csv", index=False)
pd.DataFrame([dict(d, layer=k) for k, v in dropped.items() for d in v]).to_csv(f"{OUT_DIR}/m2_hospitals_off_road_{stamp}.csv", index=False)
pd.DataFrame([dict(r, layer=k) for k, v in verdicts.items() for r in v["rows"]]).to_csv(f"{OUT_DIR}/m2_rule5_{stamp}.csv", index=False)
meta = {"run": "M2 hospitals v0.1", "commit": SHA, "osm_extract_date": EXTRACT_DATE, "graph_hash": GRAPH_HASH,
        "study_bbox": STUDY, "cell_m": A.CELL_M, "snap_max_m": A.SNAP_MAX_M, "dest_snap_max_m": A.DEST_SNAP_MAX_M,
        "route_limit_m": A.ROUTE_LIMIT_M, "population": f"GHS-POP E{GHS_EPOCH} R2023A 4326 3ss, modelled; JRC, reuse with acknowledgement",
        "hospital_list": os.path.basename(TG_CSV) + " (Open Data Telangana, GODL-India)", "columns_used": COL,
        "hospitals": {k: len(v) for k, v in layers.items()}, "verdicts": {k: {kk: vv for kk, vv in v.items() if kk != "rows"} for k, v in verdicts.items()},
        "status": "CANDIDATES. Not for publication before the verification gate."}
json_path = f"{OUT_DIR}/m2_meta_{stamp}.json"
import json; json.dump(meta, open(json_path, "w"), indent=2, default=str)
z = shutil.make_archive(f"/content/arcroad_m2_hospitals_{stamp}", "zip", OUT_DIR)
print("wrote", z)
from google.colab import files; files.download(z)